# V0 template fit

## Setup

In [ ]:
import hipe4ml as h4ml
import uproot
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from hipe4ml.tree_handler import TreeHandler
import pandas as pd
import Utils
import importlib
from scipy.stats import ks_2samp
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster
from iminuit import Minuit
import TemplateFitLibrary

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

In [ ]:
importlib.reload(TemplateFitLibrary)
importlib.reload(Utils)

In [ ]:
df = Utils.get_dataframe(["Data/ppDataLHC26ag.root","Data/LHC25l1_ppMC.root"], folder_name="DF_*")
df = Utils.process_dataframe(df, makedummies=False)

## Execution

In [ ]:
fitfeature = "DeltaPhi"  # Feature to fit
nbins = 50
starting_frac = 0.25
rangeovalues = (-3, 3)  # Range of values for the fit

### Total Fit

In [ ]:
# 1. Build the MC templates.
templates = TemplateFitLibrary.build_templates(
    mc_a=df[df['MatchLabel'].isin(Utils.MATCH_LABEL_GROUPS["True"])],
    mc_b=df[df['MatchLabel'].isin(Utils.MATCH_LABEL_GROUPS["Non-True"])],
    feature=fitfeature,
    bins=nbins,
    value_range=rangeovalues
)

# 2. Fit the real-data distribution.
result = TemplateFitLibrary.fit_template_mixture(
    data=df[df['MatchLabel'].isin(Utils.MATCH_LABEL_GROUPS["Data"])],
    templates=templates,
    initial_fraction=starting_frac,
)

# 3. Print the fit summary.
print(result["minuit"])

print(f"\nClass A fraction: {result['fraction_a']:.4f}"
      f" +/- {result['fraction_a_error']:.4f}")

print(f"Class B fraction: {result['fraction_b']:.4f}"
      f" +/- {result['fraction_b_error']:.4f}")

print(f"Data entries in fit range: {result['n_data']}")
print(f"Valid data entries outside fit range: {result['n_outside']}")

# 4. Plot the result.
TemplateFitLibrary.plot_template_fit(result, templates)

### Leader fit

In [ ]:
df_leading = df.loc[df.groupby("mchID")["Chi2Match"].idxmin()].reset_index(drop=True)

In [ ]:
# 1. Build the MC templates.
templates = TemplateFitLibrary.build_templates(
    mc_a=df_leading[df_leading['MatchLabel'].isin(Utils.MATCH_LABEL_GROUPS["True"])],
    mc_b=df_leading[df_leading['MatchLabel'].isin(Utils.MATCH_LABEL_GROUPS["Non-True"])],
    feature=fitfeature,
    bins=nbins,
    value_range=rangeovalues
)

# 2. Fit the real-data distribution.
result = TemplateFitLibrary.fit_template_mixture(
    data=df_leading[df_leading['MatchLabel'].isin(Utils.MATCH_LABEL_GROUPS["Data"])],
    templates=templates,
    initial_fraction=starting_frac,
)

# 3. Print the fit summary.
print(result["minuit"])

print(f"\nClass A fraction: {result['fraction_a']:.4f}"
      f" +/- {result['fraction_a_error']:.4f}")

print(f"Class B fraction: {result['fraction_b']:.4f}"
      f" +/- {result['fraction_b_error']:.4f}")

print(f"Data entries in fit range: {result['n_data']}")
print(f"Valid data entries outside fit range: {result['n_outside']}")

# 4. Plot the result.
TemplateFitLibrary.plot_template_fit(result, templates)